In [1]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-27

In [2]:
from pyhydra.climate.spatial_analysis import (
    FloodEventCopula,
    fit_marginal,
    fit_discrete_marginal,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Multivariate Flood Event Copulas

This notebook demonstrates how to use the `FloodEventCopula` class and its
helper functions to model and sample synthetic flood events that are
characterised by multiple correlated variables.

**Typical use case:** given a catalogue of observed flood events with peak
discharge, volume, duration, and season — fit marginal distributions to each
variable, connect them through a Gaussian copula, and sample an ensemble of
synthetic events preserving the observed dependence structure.

```bash
conda install -c conda-forge openturns
```

| Class / Function | Purpose |
|------------------|---------|
| `fit_marginal` | BIC-optimal marginal for a continuous variable |
| `fit_discrete_marginal` | Empirical distribution for an integer variable |
| `FloodEventCopula` | Full multivariate model: marginals + Gaussian copula |

---
## Synthetic flood catalogue

We simulate 80 flood events with four characterising variables:

| Variable | Type | Units | Distribution |
|----------|------|-------|--------------|
| `Qmax` | continuous | m³/s | LogNormal |
| `volume` | continuous | hm³ | LogNormal |
| `duration` | continuous | days | Gamma |
| `season` | discrete | 1–4 | Empirical |

Strong correlation between Qmax and volume (physically expected).

In [3]:
rng = np.random.default_rng(0)

n = 80

# Correlated log-normal base variables (Qmax, volume)
cov = np.array([[1.0, 0.85], [0.85, 1.0]])
Z = rng.multivariate_normal([0, 0], cov, size=n)
Qmax   = np.exp(6.5 + 0.6 * Z[:, 0])   # mean ≈ 660 m³/s
volume = np.exp(3.2 + 0.5 * Z[:, 1])   # mean ≈ 25 hm³

# Duration correlated with volume
duration = np.clip(volume / 4 + rng.exponential(2, n), 0.5, None)

# Season: winter (1) more likely for high flows
season = rng.choice([1, 2, 3, 4], size=n, p=[0.45, 0.25, 0.20, 0.10])

events = pd.DataFrame({
    "Qmax":     Qmax,
    "volume":   volume,
    "duration": duration,
    "season":   season,
})

print(events.describe().round(1))

         Qmax  volume  duration  season
count    80.0    80.0      80.0    80.0
mean    812.7    28.9       9.4     2.1
std     503.6    14.8       4.1     1.1
min     189.5     8.5       2.9     1.0
25%     464.7    20.1       6.3     1.0
50%     665.0    25.6       8.5     2.0
75%     989.8    33.6      11.9     3.0
max    2637.6    90.5      23.7     4.0


In [4]:
# Observed pairwise scatter
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].scatter(events.Qmax, events.volume, alpha=0.6, c=events.season, cmap="Set1", s=25)
axes[0].set_xlabel("Qmax (m³/s)"); axes[0].set_ylabel("Volume (hm³)")
axes[1].scatter(events.Qmax, events.duration, alpha=0.6, c=events.season, cmap="Set1", s=25)
axes[1].set_xlabel("Qmax (m³/s)"); axes[1].set_ylabel("Duration (days)")
axes[2].scatter(events.volume, events.duration, alpha=0.6, c=events.season, cmap="Set1", s=25)
axes[2].set_xlabel("Volume (hm³)"); axes[2].set_ylabel("Duration (days)")
for ax in axes:
    ax.grid(alpha=0.3)
plt.suptitle("Observed flood catalogue — pairwise scatter", fontsize=12)
plt.tight_layout()
plt.show()

---
## 1. Fitting marginal distributions

`fit_marginal` tries a pool of openturns distributions and selects the one
with the lowest BIC.

In [5]:
try:
    # Continuous marginals
    marg_qmax, _   = fit_marginal(events["Qmax"].values,   variable_name="Qmax")
    marg_vol, _    = fit_marginal(events["volume"].values,  variable_name="volume")
    marg_dur, _    = fit_marginal(events["duration"].values, variable_name="duration")

    print("Best-fit marginals:")
    for name, m in [("Qmax", marg_qmax), ("volume", marg_vol), ("duration", marg_dur)]:
        print(f"  {name:<10}: {m}")

    # Discrete marginal for season
    marg_season, _ = fit_discrete_marginal(events["season"].values)
    print(f"  season    : UserDefined (empirical)")

except ImportError:
    print("(openturns not installed — conda install -c conda-forge openturns)")

Best marginal [Qmax]: InverseNormal(mu = 812.669, lambda = 1979.68)
Best marginal [volume]: InverseNormal(mu = 28.9134, lambda = 125.919)
Best marginal [duration]: InverseNormal(mu = 9.38906, lambda = 51.7262)
Best-fit marginals:
  Qmax      : InverseNormal(mu = 812.669, lambda = 1979.68)
  volume    : InverseNormal(mu = 28.9134, lambda = 125.919)
  duration  : InverseNormal(mu = 9.38906, lambda = 51.7262)
  season    : UserDefined (empirical)


In [6]:
# Plot observed histogram vs fitted marginal for Qmax
try:
    import openturns as ot
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, (name, data, marg) in zip(axes, [
        ("Qmax (m³/s)",     events.Qmax.values,     marg_qmax),
        ("Volume (hm³)",    events.volume.values,   marg_vol),
        ("Duration (days)", events.duration.values, marg_dur),
    ]):
        x = np.linspace(data.min(), data.max(), 200)
        ax.hist(data, bins=20, density=True, alpha=0.5, color="steelblue", label="Observed")
        pdf = [marg.computePDF([xi]) for xi in x]
        ax.plot(x, pdf, "r-", lw=2, label=marg.getName())
        ax.set_xlabel(name)
        ax.set_ylabel("Density")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    plt.suptitle("Fitted marginal distributions", fontsize=12)
    plt.tight_layout()
    plt.show()
except (ImportError, NameError):
    print("(Marginal plot requires openturns)")

---
## 2. Fitting the full copula model

In [7]:
try:
    copula_model = FloodEventCopula(
        continuous_vars=["Qmax", "volume", "duration"],
        discrete_vars=["season"],
    )
    copula_model.fit(events)
    print("Model fitted successfully.")
    print("Correlation matrix (Gaussian copula):")
    print(copula_model.plot_dependence)
except ImportError:
    print("(openturns not installed — conda install -c conda-forge openturns)")

Best marginal [Qmax]: InverseNormal(mu = 812.669, lambda = 1979.68)
Best marginal [volume]: InverseNormal(mu = 28.9134, lambda = 125.919)
Best marginal [duration]: InverseNormal(mu = 9.38906, lambda = 51.7262)
Fitted copula: NormalCopula(R = [[  1         0.829531  0.771906 -0.112755 ]
 [  0.829531  1         0.893252 -0.159637 ]
 [  0.771906  0.893252  1        -0.100991 ]
 [ -0.112755 -0.159637 -0.100991  1        ]])
Model fitted successfully.
Correlation matrix (Gaussian copula):
<bound method FloodEventCopula.plot_dependence of <pyhydra.climate.spatial_analysis.copulas.FloodEventCopula object at 0x7fffb3d64350>>


---
## 3. Sampling synthetic events

In [8]:
try:
    synthetic = copula_model.sample(n_samples=1000)
    print(f"Synthetic catalogue: {synthetic.shape}")
    print(synthetic.describe().round(1))
except (ImportError, NameError):
    print("(Sampling requires openturns)")

Synthetic catalogue: (1000, 4)
         Qmax  volume  duration  season
count  1000.0  1000.0    1000.0  1000.0
mean    836.4    29.2       9.4     2.1
std     543.7    13.6       3.9     1.1
min     135.5     6.8       2.7     1.0
25%     463.7    18.7       6.5     1.0
50%     699.3    26.4       8.7     2.0
75%    1050.1    36.6      11.5     3.0
max    4606.2   117.7      25.4     4.0


In [9]:
# Observed vs synthetic scatter
try:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    pairs = [("Qmax", "volume"), ("Qmax", "duration"), ("volume", "duration")]
    for ax, (xv, yv) in zip(axes, pairs):
        ax.scatter(synthetic[xv], synthetic[yv], alpha=0.2, s=8,
                   color="steelblue", label="Synthetic")
        ax.scatter(events[xv], events[yv], alpha=0.8, s=25,
                   color="tomato", edgecolors="k", linewidths=0.5, label="Observed")
        ax.set_xlabel(xv)
        ax.set_ylabel(yv)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    plt.suptitle("Observed vs synthetic flood events", fontsize=12)
    plt.tight_layout()
    plt.show()
except (ImportError, NameError):
    print("(Plot requires openturns)")

---
## 4. Visualise copula dependence

In [10]:
try:
    copula_model.plot_marginals()
except (ImportError, NameError):
    print("(Requires openturns)")

In [11]:
try:
    copula_model.plot_dependence(observed=events, synthetic=synthetic)
except (ImportError, NameError):
    print("(Requires openturns)")

---
## 5. Joint exceedance probabilities

Estimate the probability that both Qmax and volume exceed given thresholds
simultaneously — a quantity often needed in dam safety analysis.

In [12]:
try:
    # Use synthetic sample as a Monte Carlo estimator
    Qmax_thresh  = np.percentile(events.Qmax,   [50, 75, 90, 95])
    vol_thresh   = np.percentile(events.volume, [50, 75, 90, 95])

    print("P(Qmax > q AND volume > v):")
    print(f"{'Return period (T)':>20}  {'Q-threshold':>12}  {'V-threshold':>12}  {'P(joint)':>10}")
    for q, v, p_label in zip(Qmax_thresh, vol_thresh, ["2y", "4y", "10y", "20y"]):
        p_joint = ((synthetic.Qmax > q) & (synthetic.volume > v)).mean()
        print(f"{p_label:>20}  {q:>12.0f}  {v:>12.1f}  {p_joint:>10.4f}")

except (ImportError, NameError):
    # Demonstrate with the synthetic data we already have
    Qmax_thresh = np.percentile(events.Qmax,   [50, 75, 90, 95])
    vol_thresh  = np.percentile(events.volume, [50, 75, 90, 95])
    print("(Approximate with observed catalogue — install openturns for Monte Carlo)")
    print(f"{'Percentile':>12}  {'Q (m³/s)':>10}  {'V (hm³)':>10}")
    for pct, q, v in zip([50, 75, 90, 95], Qmax_thresh, vol_thresh):
        print(f"{pct:>12}  {q:>10.0f}  {v:>10.1f}")

P(Qmax > q AND volume > v):
   Return period (T)   Q-threshold   V-threshold    P(joint)
                  2y           665          25.6      0.4300
                  4y           990          33.6      0.2180
                 10y          1451          44.8      0.0790
                 20y          1959          55.6      0.0190


---
## 6. Season-conditional generation

Because `season` is included as a discrete variable in the copula, the sampled
events already carry season labels that are statistically consistent with the
observed dependence between season and flood magnitude.

In [13]:
try:
    season_labels = {1: "Winter", 2: "Spring", 3: "Summer", 4: "Autumn"}

    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ["royalblue", "forestgreen", "tomato", "darkorange"]
    for s, (sn, col) in enumerate(zip(season_labels.values(), colors), start=1):
        mask = synthetic.season.round().astype(int) == s
        ax.scatter(synthetic.Qmax[mask], synthetic.volume[mask],
                   alpha=0.3, s=8, color=col, label=f"{sn} (n={mask.sum()})")
    ax.scatter(events.Qmax, events.volume, c="k", s=30, zorder=5,
               edgecolors="white", linewidths=0.5, label="Observed")
    ax.set_xlabel("Qmax (m³/s)")
    ax.set_ylabel("Volume (hm³)")
    ax.legend(fontsize=9, markerscale=2)
    ax.set_title("Synthetic events coloured by season", fontsize=12)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

except (ImportError, NameError):
    # Observed events only
    season_labels = {1: "Winter", 2: "Spring", 3: "Summer", 4: "Autumn"}
    colors = ["royalblue", "forestgreen", "tomato", "darkorange"]
    fig, ax = plt.subplots(figsize=(8, 5))
    for s, (sn, col) in enumerate(zip(season_labels.values(), colors), start=1):
        mask = events.season == s
        ax.scatter(events.Qmax[mask], events.volume[mask],
                   s=40, color=col, label=f"{sn} (n={mask.sum()})")
    ax.set_xlabel("Qmax (m³/s)")
    ax.set_ylabel("Volume (hm³)")
    ax.legend()
    ax.set_title("Observed events by season", fontsize=12)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()